In [1]:
from pyspark.sql import functions as F

# Testdaten nur im Arbeitsspeicher
test_data = [
    ("T001", "ECU_Engine", "PASS"),
    ("T002", "ECU_Brake", "FAIL"),
    ("T003", "ECU_Battery", "PASS"),
    ("T003", "ECU_Battery", "PASS"),
    (None, "ECU_Engine", "ERROR")
]

columns = ["TestID", "ECU", "TestResult"]

df_test = spark.createDataFrame(
    test_data, columns
)

display(df_test)


StatementMeta(, 31abe5d8-491f-4319-9f09-ab307d99e8d4, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a95c4ae9-a44a-4b7c-b4f6-9deeb21a86b3)

In [2]:
from pyspark.sql import functions as F

# Qualitätsprüfungen auf dem Test-DataFrame
total_rows = df_test.count()

null_ids = df_test.filter(
    F.col("TestID").isNull()
).count()

duplicate_ids = (
    df_test
    .filter(F.col("TestID").isNotNull())
    .groupBy("TestID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_results = df_test.filter(
    F.col("TestResult").isNull() |
    (~F.upper(F.trim(F.col("TestResult")))
      .isin("PASS", "FAIL"))
).count()

quality_status = (
    "PASS"
    if null_ids == 0
    and duplicate_ids == 0
    and invalid_results == 0
    else "FAIL"
)

print("Datensätze:", total_rows)
print("Fehlende TestIDs:", null_ids)
print("Doppelte TestIDs:", duplicate_ids)
print("Ungültige Testergebnisse:", invalid_results)
print("Data Quality Status:", quality_status)


StatementMeta(, 31abe5d8-491f-4319-9f09-ab307d99e8d4, 4, Finished, Available, Finished, False)

Datensätze: 5
Fehlende TestIDs: 1
Doppelte TestIDs: 1
Ungültige Testergebnisse: 1
Data Quality Status: FAIL


In [3]:
# Data Quality Gate – Negativtest

if quality_status == "FAIL":
    raise ValueError(
        "DATA QUALITY FAILED: "
        f"MissingTestIDs={null_ids}, "
        f"DuplicateTestIDs={duplicate_ids}, "
        f"InvalidResults={invalid_results}"
    )

print("DATA QUALITY PASSED - Verarbeitung freigegeben.")


StatementMeta(, 31abe5d8-491f-4319-9f09-ab307d99e8d4, 5, Finished, Available, Finished, False)

ValueError: DATA QUALITY FAILED: MissingTestIDs=1, DuplicateTestIDs=1, InvalidResults=1

## Data Quality – Negative Test

### Objective
Verify that invalid automotive diagnostic data
is detected and rejected by the quality gate.

### Test Dataset
- Total records: 5
- Missing TestIDs: 1
- Duplicate TestIDs: 1
- Invalid test results: 1

### Expected Result
QualityStatus = FAIL

### Actual Result
QualityStatus = FAIL

### Quality Gate
A ValueError is raised when validation fails.

### Test Conclusion
PASS – Invalid data was successfully detected
and the notebook raised the expected exception.

### Scope
This test validates the notebook quality gate.
Pipeline-level failure handling is tested separately.
